## CELL 1 — Config

In [1]:
"""
NOTEBOOK 2A.1 - Integrated Gradients (Komponen 1 XAI)
Fase 2A IndoMeme-XAI

Atribusi per-token teks via Captum LayerIntegratedGradients.
Output dipakai utk:
  - Visualisasi case study (Fase 2C)
  - ROAR masking di teks (Fase 2B)
"""

from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input/datasets/reynaldaryansyah")
ROOT_DIR     = KAGGLE_INPUT / "xai-dataset" / "data" / "data"
CSV_PATH     = ROOT_DIR / "processed" / "all_splits.csv"
IMG_DIR      = ROOT_DIR / "images"

# Path checkpoint frozen dari 1B.9 seed=7. Sesuaikan jika nama dataset Kaggle berbeda.
CKPT_PATH    = Path("/kaggle/input/datasets/reynaldaryansyah/xai-dataset/exp-1b9-ensemble/model_seed7.pt")

OUTPUT_DIR = Path("/kaggle/working/xai_2A1_integrated_gradients")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = dict(
    exp_id           = "2A.1_integrated_gradients",
    seed             = 42,                  # untuk reproducibility, model sudah di-freeze

    text_model_name  = "indolem/indobertweet-base-uncased",
    image_model_name = "google/vit-base-patch16-224-in21k",
    freeze_n_layers  = 4,
    topic_emb_dim    = 32,
    head_hidden      = 256,
    dropout          = 0.3,
    icam_num_heads   = 8,
    icam_dropout     = 0.1,

    # SupCon proj (dibutuhkan agar load_state_dict cocok)
    supcon_proj_hidden     = 128,
    supcon_proj_out        = 128,

    max_seq_len      = 128,

    # IG params
    ig_n_steps       = 50,
    ig_internal_bs   = 8,                   # internal batch utk IG steps (mengontrol memory)
    ig_outer_bs      = 4,                   # outer batch utk samples
    ig_target_mode   = "predicted",         # "predicted" atau "hate" (=1)

    # Subset utk run (None = full test set)
    subset_size      = None,                # set ke 100 utk quick smoke test
)

print(f"Experiment: {CONFIG['exp_id']}")
print(f"Checkpoint: {CKPT_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"IG params : n_steps={CONFIG['ig_n_steps']}, internal_bs={CONFIG['ig_internal_bs']}")

Experiment: 2A.1_integrated_gradients
Checkpoint: /kaggle/input/datasets/reynaldaryansyah/xai-dataset/exp-1b9-ensemble/model_seed7.pt
Output dir: /kaggle/working/xai_2A1_integrated_gradients
IG params : n_steps=50, internal_bs=8


## CELL 2 — Install Captum + Imports

In [2]:
# Captum mungkin belum ter-install di Kaggle base image
import subprocess, sys
def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import captum
except ImportError:
    pip_install("captum")
    import captum
print(f"Captum version: {captum.__version__}")

import os, json, random, time, math, re, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from torchvision import transforms
from transformers import AutoTokenizer, AutoModel, ViTImageProcessor, ViTModel
from captum.attr import LayerIntegratedGradients
from tqdm.auto import tqdm

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 13.2 MB/s eta 0:00:00
Captum version: 0.9.0
Device: cuda | PyTorch: 2.10.0+cu128


## CELL 3 — Slang Dictionary

In [3]:
SLANG_DICT = {
    "gw":"saya","gue":"saya","gua":"saya","guwe":"saya",
    "lu":"kamu","lo":"kamu","loe":"kamu","elo":"kamu","elu":"kamu",
    "ane":"saya","ente":"kamu","agan":"kamu","sis":"kakak",
    "bro":"kawan","bray":"kawan","bre":"kawan","cuy":"kawan",
    "klian":"kalian","kmu":"kamu","kamuh":"kamu",
    "aq":"aku","akoh":"aku","akuh":"aku",
    "gak":"tidak","ga":"tidak","gk":"tidak","nggak":"tidak",
    "ngga":"tidak","ngk":"tidak","kgk":"tidak","tdk":"tidak",
    "engga":"tidak","enggak":"tidak","kga":"tidak",
    "yg":"yang","dgn":"dengan","dg":"dengan","krn":"karena",
    "krna":"karena","utk":"untuk","tp":"tapi","tpi":"tapi",
    "klo":"kalau","klau":"kalau","kl":"kalau","klu":"kalau",
    "spy":"supaya","ato":"atau",
    "bgt":"banget","bgtt":"banget","bngt":"banget",
    "bnr":"benar","bener":"benar",
    "emg":"memang","emang":"memang","mmg":"memang",
    "skrg":"sekarang","skrng":"sekarang",
    "td":"tadi","tdi":"tadi","bsk":"besok",
    "knp":"kenapa","gmn":"bagaimana","gimana":"bagaimana",
    "gmana":"bagaimana","kyk":"seperti","kayak":"seperti","kek":"seperti",
    "udh":"sudah","udah":"sudah","uda":"sudah","dah":"sudah",
    "blm":"belum","blom":"belum",
    "lg":"lagi","lgi":"lagi","msh":"masih","msi":"masih",
    "jd":"jadi","jdi":"jadi",
    "tau":"tahu","taw":"tahu","tw":"tahu",
    "liat":"lihat","ngeliat":"melihat","denger":"dengar",
    "ngomong":"bicara","blng":"bilang","bilg":"bilang",
    "aja":"saja","doang":"saja","doank":"saja",
    "sih":"","deh":"","dong":"","kok":"","lah":"",
    "loh":"","lho":"","yah":"",
    "iy":"iya","iyaa":"iya","iye":"iya","yup":"iya","yoi":"iya",
    "org":"orang","orng":"orang","orgnya":"orangnya",
    "dr":"dari","dri":"dari","kpd":"kepada",
    "thd":"terhadap","ttg":"tentang",
    "abis":"habis","trs":"terus","trus":"terus",
    "tmn":"teman","temen":"teman","tman":"teman",
    "rmh":"rumah","kntr":"kantor","krj":"kerja",
    "ank":"anak","indo":"indonesia",
    "cwo":"cowok","cwe":"cewek","cwok":"cowok","cwek":"cewek",
    "wkwk":"haha","wkwkwk":"haha","wkwkwkwk":"haha","wkwkwkwkwk":"haha",
    "hahaha":"haha","hahahaha":"haha",
    "hehehe":"hehe","hehehehe":"hehe",
    "anjir":"anjing","anjay":"anjing","anjrit":"anjing",
    "anjr":"anjing","anying":"anjing","njir":"anjing",
    "bgsd":"bangsat","bgst":"bangsat",
    "bjir":"anjing","bjg":"bajingan",
    "kntl":"kontol","mmk":"memek","tai":"tahi",
}

## CELL 4 — clean_ocr

In [4]:
URL_PATTERN     = re.compile(r"(https?://\S+|www\.\S+|\S+\.(?:com|co\.id|net|org|id|tv|me)\b\S*)", re.IGNORECASE)
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
ALLOWED_PUNCT   = re.compile(r"[^\w\s\.,!?\-'\"]+", re.UNICODE)
MULTI_SPACE     = re.compile(r"\s+")

def _normalize_unicode(t): return unicodedata.normalize("NFKC", t)

def _normalize_slang(text, slang_dict=SLANG_DICT):
    out = []
    for tok in text.split():
        prefix, suffix, core = "", "", tok
        while core and not core[0].isalnum(): prefix += core[0]; core = core[1:]
        while core and not core[-1].isalnum(): suffix = core[-1] + suffix; core = core[:-1]
        out.append(f"{prefix}{slang_dict.get(core.lower(), core)}{suffix}")
    return " ".join(out)

def clean_ocr(text):
    if text is None: return ""
    if isinstance(text, float) and pd.isna(text): return ""
    text = _normalize_unicode(str(text)).lower()
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(r"\1", text)
    text = ALLOWED_PUNCT.sub(" ", text)
    text = _normalize_slang(text)
    return MULTI_SPACE.sub(" ", text).strip()

## CELL 5 — format_input

In [5]:
SEP_TOKEN = "[SEP]"

def _safe_str(x):
    if x is None: return ""
    if isinstance(x, float) and pd.isna(x): return ""
    return str(x).strip()

def format_input(topic, caption, ocr_cleaned, sep=SEP_TOKEN):
    if topic is None or (isinstance(topic, float) and pd.isna(topic)):
        topic_str = "unknown"
    elif isinstance(topic, (list, tuple, set)):
        items = sorted({str(t).lower().strip() for t in topic if t})
        topic_str = ",".join(items) if items else "unknown"
    else:
        topic_str = str(topic).lower().strip() or "unknown"
    return f"[{topic_str}] {_safe_str(caption)} {sep} {_safe_str(ocr_cleaned)}".strip()

## CELL 6 — Load Data + Topic Encoding

In [6]:
df = pd.read_csv(CSV_PATH)
LABEL_MAP = {"not hate": 0, "hate": 1}
df["label"] = df["hate_final"].map(LABEL_MAP)
df["ocr_cleaned"] = df["ocr"].apply(clean_ocr)
df["text_input"]  = df.apply(lambda r: format_input(r["topic"], r["caption"], r["ocr_cleaned"]), axis=1)

def resolve_image_path(row):
    raw = row.get("local_image_path") or row.get("image_path")
    if raw is None or (isinstance(raw, float) and pd.isna(raw)): return None
    fname = Path(str(raw).replace("\\","/")).name
    return str(IMG_DIR / row["split"] / fname)
df["full_image_path"] = df.apply(resolve_image_path, axis=1)

TOPIC_LIST = ["political","national origin","individual","institutions",
              "gender","religion","social sub-groups","unknown"]
TOPIC2IDX = {t: i for i, t in enumerate(TOPIC_LIST)}
N_TOPICS  = len(TOPIC_LIST)

def topic_to_multihot(topic):
    vec = np.zeros(N_TOPICS, dtype=np.float32)
    if topic is None or (isinstance(topic, float) and pd.isna(topic)):
        vec[TOPIC2IDX["unknown"]] = 1.0; return vec
    parts = [p.strip().lower() for p in str(topic).split(",") if p.strip()]
    found = False
    for p in parts:
        if p in TOPIC2IDX: vec[TOPIC2IDX[p]] = 1.0; found = True
    if not found: vec[TOPIC2IDX["unknown"]] = 1.0
    return vec

df["topic_multihot"] = df["topic"].apply(topic_to_multihot)
df_test = df[df["split"] == "test"].copy().reset_index(drop=True)

# Optional: subset utk smoke test
if CONFIG["subset_size"] is not None:
    df_test = df_test.head(CONFIG["subset_size"]).reset_index(drop=True)

print(f"Test samples: {len(df_test)}")
print(f"Label dist : {df_test['label'].value_counts().to_dict()}")

Test samples: 1005
Label dist : {0: 658, 1: 347}


## CELL 7 — Tokenizer & Image Processor + Transforms

In [7]:
tokenizer       = AutoTokenizer.from_pretrained(CONFIG["text_model_name"])
image_processor = ViTImageProcessor.from_pretrained(CONFIG["image_model_name"])

IMG_MEAN = image_processor.image_mean
IMG_STD  = image_processor.image_std
IMG_SIZE = image_processor.size["height"] if isinstance(image_processor.size, dict) else 224

EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])
print(f"Tokenizer pad_token_id: {tokenizer.pad_token_id}, sep: {tokenizer.sep_token!r}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Tokenizer pad_token_id: 0, sep: '[SEP]'


## CELL 8 — Dataset Class 

In [8]:
class IndoMemeDataset(Dataset):
    def __init__(self, df_split, tokenizer, transform, max_seq_len=128):
        self.df          = df_split.reset_index(drop=True)
        self.tokenizer   = tokenizer
        self.transform   = transform
        self.max_seq_len = max_seq_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["full_image_path"]).convert("RGB")
        img_tensor = self.transform(img)
        enc = self.tokenizer(row["text_input"], padding="max_length", truncation=True,
                             max_length=self.max_seq_len, return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "pixel_values":   img_tensor,
            "topic":          torch.from_numpy(row["topic_multihot"]),
            "label":          torch.tensor(int(row["label"]), dtype=torch.long),
            "idx":            int(row["idx"]),
        }

ds_test = IndoMemeDataset(df_test, tokenizer, EVAL_TRANSFORM, CONFIG["max_seq_len"])
dl_test = DataLoader(ds_test, batch_size=CONFIG["ig_outer_bs"], shuffle=False,
                     num_workers=2, pin_memory=True)
print(f"Loader: {len(dl_test)} batches of {CONFIG['ig_outer_bs']} samples")

Loader: 252 batches of 4 samples


## CELL 9 — ICAM Module

In [9]:
class ICAMFusion(nn.Module):
    def __init__(self, hidden_dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        self.t2i_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.i2t_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.gate_relevant    = nn.Linear(2 * hidden_dim, hidden_dim)
        self.gate_incongruent = nn.Linear(2 * hidden_dim, hidden_dim)
        self.fusion_proj = nn.Linear(hidden_dim, hidden_dim)
        self.layer_norm  = nn.LayerNorm(hidden_dim)
        self.save_attn = False

    def forward(self, text_h, image_h, text_mask=None):
        text_kpm = (~text_mask.bool()) if text_mask is not None else None
        t2i_out, _ = self.t2i_attn(text_h, image_h, image_h,
                                    key_padding_mask=None,
                                    need_weights=self.save_attn, average_attn_weights=True)
        i2t_out, _ = self.i2t_attn(image_h, text_h, text_h,
                                    key_padding_mask=text_kpm,
                                    need_weights=self.save_attn, average_attn_weights=True)
        h_image_pooled = i2t_out.mean(dim=1)
        text_norm  = F.normalize(text_h,  dim=-1)
        image_norm = F.normalize(image_h, dim=-1)
        sim         = torch.bmm(text_norm, image_norm.transpose(1, 2))
        incongruity = 1.0 - sim.abs()
        inc_w       = F.softmax(incongruity, dim=-1)
        h_inc       = torch.bmm(inc_w, image_h)
        if text_mask is not None:
            m = text_mask.unsqueeze(-1).float()
            h_inc_pooled = (h_inc * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)
        else:
            h_inc_pooled = h_inc.mean(dim=1)
        h_text = text_h[:, 0]
        g1 = torch.sigmoid(self.gate_relevant(   torch.cat([h_text, h_image_pooled], dim=-1)))
        g2 = torch.sigmoid(self.gate_incongruent(torch.cat([h_text, h_inc_pooled  ], dim=-1)))
        fused   = h_text + g1 * h_image_pooled + g2 * h_inc_pooled
        h_fused = self.layer_norm(self.fusion_proj(fused))
        return h_fused, g1.mean().item(), g2.mean().item()

## CELL 10 — Full Model

In [10]:
class ICAMHateModelSupCon(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.text_encoder  = AutoModel.from_pretrained(cfg["text_model_name"])
        self.image_encoder = ViTModel.from_pretrained(cfg["image_model_name"])
        for p in self.text_encoder.parameters():  p.requires_grad = False
        for p in self.image_encoder.parameters(): p.requires_grad = False

        hidden = self.text_encoder.config.hidden_size
        self.icam = ICAMFusion(hidden_dim=hidden, num_heads=cfg["icam_num_heads"], dropout=cfg["icam_dropout"])
        self.supcon_proj = nn.Sequential(
            nn.Linear(hidden, cfg["supcon_proj_hidden"]),
            nn.ReLU(),
            nn.Linear(cfg["supcon_proj_hidden"], cfg["supcon_proj_out"]),
        )
        self.topic_proj = nn.Sequential(nn.Linear(N_TOPICS, cfg["topic_emb_dim"]), nn.GELU())
        head_in = hidden + cfg["topic_emb_dim"]
        self.head = nn.Sequential(
            nn.Linear(head_in, cfg["head_hidden"]), nn.GELU(),
            nn.Dropout(cfg["dropout"]),
            nn.Linear(cfg["head_hidden"], 2),
        )

    def forward(self, input_ids, attention_mask, pixel_values, topic):
        t_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        i_out = self.image_encoder(pixel_values=pixel_values)
        h_fused, _, _ = self.icam(t_out.last_hidden_state, i_out.last_hidden_state, attention_mask)
        topic_emb = self.topic_proj(topic)
        logits    = self.head(torch.cat([h_fused, topic_emb], dim=-1))
        return logits  # NB: hanya return logits utk Captum

## CELL 11 — Load Frozen Checkpoint

In [11]:
model = ICAMHateModelSupCon(CONFIG).to(DEVICE)

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
# Cek format: kalau hanya state_dict, load langsung; kalau dict dgn key 'state_dict', extract
if isinstance(ckpt, dict) and "state_dict" in ckpt:
    state = ckpt["state_dict"]
    print(f"Checkpoint epoch: {ckpt.get('epoch', '?')}")
else:
    state = ckpt

# Karena Model di sini return logits saja, sedangkan training pakai 4-tuple,
# state_dict tetap kompatibel (parameter sama, hanya forward berbeda).
missing, unexpected = model.load_state_dict(state, strict=False)
print(f"Missing keys (non-fatal kalau hanya buffer): {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")

# Final freeze: semua param requires_grad=False; eval mode
for p in model.parameters(): p.requires_grad = False
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Frozen model loaded. Total params: {n_params/1e6:.2f}M (all frozen)")

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Checkpoint epoch: 15
Missing keys (non-fatal kalau hanya buffer): 0
Unexpected keys: 0
Frozen model loaded. Total params: 204.95M (all frozen)


## CELL 12 — Captum Setup

In [12]:
# LayerIntegratedGradients dengan target = embedding layer text_encoder
text_embed_layer = model.text_encoder.embeddings

lig = LayerIntegratedGradients(
    forward_func=model,             # forward(input_ids, attention_mask, pixel_values, topic) -> logits
    layer=text_embed_layer,
)

print(f"LIG configured. Target layer: BertEmbeddings ({sum(p.numel() for p in text_embed_layer.parameters())/1e6:.2f}M params)")

LIG configured. Target layer: BertEmbeddings (24.91M params)


## CELL 13 — IG Batch Runner

In [13]:
def run_ig_batch(batch, lig, n_steps, internal_bs, target_mode="predicted"):
    """
    Returns dict per sample:
      - token_l2:    [T] - L2 norm of attribution per token
      - token_sum:   [T] - signed sum across embedding dim
      - tokens:      list[str] - decoded tokens
      - pred_class:  int (0/1)
      - pred_prob:   float - softmax prob of predicted class
      - true_label:  int
      - delta:       float - convergence delta
      - idx:         int - sample id
    """
    input_ids      = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    pixel_values   = batch["pixel_values"].to(DEVICE)
    topic          = batch["topic"].to(DEVICE)
    labels         = batch["label"]
    idxs           = batch["idx"]

    B = input_ids.size(0)
    baseline_input_ids = torch.full_like(input_ids, tokenizer.pad_token_id)

    # Predict first
    with torch.no_grad():
        logits = model(input_ids, attention_mask, pixel_values, topic)
        probs  = F.softmax(logits, dim=-1)
        preds  = probs.argmax(dim=-1)

    # Pilih target
    if target_mode == "predicted":
        targets = preds
    elif target_mode == "hate":
        targets = torch.full((B,), 1, device=DEVICE, dtype=torch.long)
    else:
        raise ValueError(f"Unknown target_mode: {target_mode}")

    # Compute IG attribution
    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_input_ids,
        additional_forward_args=(attention_mask, pixel_values, topic),
        target=targets,
        n_steps=n_steps,
        internal_batch_size=internal_bs,
        return_convergence_delta=True,
    )
    # attributions: [B, T, 768]
    token_l2  = attributions.norm(dim=-1).cpu().numpy()    # [B, T]
    token_sum = attributions.sum(dim=-1).cpu().numpy()     # [B, T]

    # Decode tokens per sample (untuk visualisasi)
    results = []
    for i in range(B):
        ids_i = input_ids[i].cpu().tolist()
        toks  = tokenizer.convert_ids_to_tokens(ids_i)
        results.append({
            "idx":         int(idxs[i]),
            "tokens":      toks,
            "token_l2":    token_l2[i].astype(np.float32),
            "token_sum":   token_sum[i].astype(np.float32),
            "attention_mask": attention_mask[i].cpu().numpy().astype(np.int8),
            "pred_class":  int(preds[i].item()),
            "pred_prob":   float(probs[i, preds[i]].item()),
            "true_label":  int(labels[i].item()),
            "delta":       float(delta[i].item()),
        })
    return results

## CELL 14 — Run IG on Test Set

In [14]:
all_results = []
t0 = time.time()

for batch in tqdm(dl_test, desc="IG attribution"):
    batch_results = run_ig_batch(
        batch, lig,
        n_steps=CONFIG["ig_n_steps"],
        internal_bs=CONFIG["ig_internal_bs"],
        target_mode=CONFIG["ig_target_mode"],
    )
    all_results.extend(batch_results)

elapsed = time.time() - t0
print(f"\nDone. {len(all_results)} samples in {elapsed/60:.1f} min  "
      f"({elapsed/len(all_results):.2f} s/sample)")

IG attribution:   0%|          | 0/252 [00:00<?, ?it/s]


Done. 1005 samples in 27.6 min  (1.65 s/sample)


## CELL 15 — Save Attribution Arrays

In [15]:
# Compact storage: stack arrays (uniform length T=128)
n = len(all_results)
T = CONFIG["max_seq_len"]

token_l2_arr  = np.zeros((n, T), dtype=np.float32)
token_sum_arr = np.zeros((n, T), dtype=np.float32)
mask_arr      = np.zeros((n, T), dtype=np.int8)
idxs_arr      = np.array([r["idx"]        for r in all_results], dtype=np.int64)
pred_arr      = np.array([r["pred_class"] for r in all_results], dtype=np.int8)
pred_prob_arr = np.array([r["pred_prob"]  for r in all_results], dtype=np.float32)
label_arr     = np.array([r["true_label"] for r in all_results], dtype=np.int8)
delta_arr     = np.array([r["delta"]      for r in all_results], dtype=np.float32)

for i, r in enumerate(all_results):
    token_l2_arr[i]  = r["token_l2"]
    token_sum_arr[i] = r["token_sum"]
    mask_arr[i]      = r["attention_mask"]

np.savez_compressed(
    OUTPUT_DIR / "ig_attributions.npz",
    token_l2=token_l2_arr,
    token_sum=token_sum_arr,
    attention_mask=mask_arr,
    idxs=idxs_arr,
    pred_class=pred_arr,
    pred_prob=pred_prob_arr,
    true_label=label_arr,
    delta=delta_arr,
)

# Tokens disimpan terpisah sbg JSON (lebih readable utk debug)
tokens_payload = [
    {"idx": int(r["idx"]), "tokens": r["tokens"]}
    for r in all_results
]
with open(OUTPUT_DIR / "ig_tokens.json", "w") as f:
    json.dump(tokens_payload, f, ensure_ascii=False)

print(f"Saved: ig_attributions.npz  (shape {token_l2_arr.shape})")
print(f"Saved: ig_tokens.json")

Saved: ig_attributions.npz  (shape (1005, 128))
Saved: ig_tokens.json


## CELL 16 — Sanity Check: Convergence Delta & Attribution Stats

In [16]:
print(f"\n{'='*72}\nIG SANITY CHECKS\n{'='*72}")
print(f"Convergence delta (lower=better):")
print(f"  mean:  {np.mean(np.abs(delta_arr)):.4f}")
print(f"  std :  {np.std(delta_arr):.4f}")
print(f"  P95 :  {np.percentile(np.abs(delta_arr), 95):.4f}")
print(f"  P99 :  {np.percentile(np.abs(delta_arr), 99):.4f}")

print(f"\nToken L2 attribution stats (only over real tokens, not pad):")
real_attr = token_l2_arr[mask_arr.astype(bool)]
print(f"  mean:  {real_attr.mean():.4f}")
print(f"  std :  {real_attr.std():.4f}")
print(f"  max :  {real_attr.max():.4f}")

print(f"\nPrediction agreement vs label (model accuracy on subset):")
acc = (pred_arr == label_arr).mean()
print(f"  accuracy: {acc:.4f}")

# Distribusi target class yg diatribusi
print(f"\nTarget distribution (pred class):")
print(f"  not_hate (0): {(pred_arr == 0).sum()}  ({(pred_arr == 0).mean()*100:.1f}%)")
print(f"  hate     (1): {(pred_arr == 1).sum()}  ({(pred_arr == 1).mean()*100:.1f}%)")


IG SANITY CHECKS
Convergence delta (lower=better):
  mean:  0.5205
  std :  1.6780
  P95 :  1.2445
  P99 :  3.5919

Token L2 attribution stats (only over real tokens, not pad):
  mean:  0.0435
  std :  0.0454
  max :  4.4660

Prediction agreement vs label (model accuracy on subset):
  accuracy: 0.8149

Target distribution (pred class):
  not_hate (0): 648  (64.5%)
  hate     (1): 357  (35.5%)


## CELL 17 — Visualisasi: Token Highlights untuk 8 Sampel Representatif

In [17]:
from IPython.display import display, HTML

def colorize(tokens, scores, mask, max_render_len=80):
    """HTML highlight tokens: red intensity = abs(score)."""
    real_scores = scores[mask.astype(bool)]
    if len(real_scores) == 0: return "(empty)"
    vmax = np.abs(real_scores).max() + 1e-8
    html = []
    n_render = 0
    for tok, sc, m in zip(tokens, scores, mask):
        if not m: continue
        if n_render >= max_render_len:
            html.append(" ...")
            break
        intensity = min(abs(sc) / vmax, 1.0)
        if sc >= 0:
            color = f"rgba(255, 60, 60, {intensity:.2f})"
        else:
            color = f"rgba(60, 60, 255, {intensity:.2f})"
        html.append(f'<span style="background-color:{color}; padding:1px 3px; margin:1px; border-radius:3px;">{tok}</span>')
        n_render += 1
    return " ".join(html)

# Pilih sampel: 2 TP (true positive hate), 2 TN (true neg), 2 FP, 2 FN
def pick(condition_mask, k):
    idx_pool = np.where(condition_mask)[0]
    return np.random.RandomState(0).choice(idx_pool, size=min(k, len(idx_pool)), replace=False)

mask_tp = (pred_arr == 1) & (label_arr == 1)
mask_tn = (pred_arr == 0) & (label_arr == 0)
mask_fp = (pred_arr == 1) & (label_arr == 0)
mask_fn = (pred_arr == 0) & (label_arr == 1)

picks = {
    "TP (hate→hate, correct)":     pick(mask_tp, 2),
    "TN (notHate→notHate, correct)": pick(mask_tn, 2),
    "FP (notHate→hate, wrong)":    pick(mask_fp, 2),
    "FN (hate→notHate, wrong)":    pick(mask_fn, 2),
}

for cat, ids in picks.items():
    display(HTML(f"<h3>{cat}</h3>"))
    for i in ids:
        r = all_results[i]
        prob_pct = r["pred_prob"] * 100
        header = (f"<b>idx={r['idx']}</b> &nbsp; pred={'hate' if r['pred_class']==1 else 'not_hate'} "
                  f"({prob_pct:.1f}%) &nbsp; true={'hate' if r['true_label']==1 else 'not_hate'}")
        body   = colorize(r["tokens"], r["token_sum"], r["attention_mask"])
        display(HTML(f"<div style='margin:6px 0;padding:6px;border-left:4px solid #888;'>"
                     f"{header}<br>{body}</div>"))

print("\nLegend: red = pushes toward predicted class | blue = pushes away")


Legend: red = pushes toward predicted class | blue = pushes away


## CELL 18 — Top-K Tokens Analysis

In [18]:
# Untuk masing-masing sampel: ambil top-5 token by L2 attribution
TOP_K = 5
top_tokens_per_sample = []
for r in all_results:
    mask = r["attention_mask"].astype(bool)
    real_scores = r["token_l2"][mask]
    real_tokens = [t for t, m in zip(r["tokens"], r["attention_mask"]) if m]
    if len(real_scores) == 0:
        top_tokens_per_sample.append([])
        continue
    top_idx = np.argsort(-real_scores)[:TOP_K]
    top_tokens_per_sample.append([(real_tokens[k], float(real_scores[k])) for k in top_idx])

# Aggregate: token paling sering muncul di top-K untuk prediksi hate
from collections import Counter
hate_top_tokens = Counter()
nothate_top_tokens = Counter()
for r, top in zip(all_results, top_tokens_per_sample):
    bag = hate_top_tokens if r["pred_class"] == 1 else nothate_top_tokens
    for tok, sc in top:
        # filter token spesial
        if tok in ("[CLS]", "[SEP]", "[PAD]", "[UNK]"): continue
        bag[tok.lower()] += 1

print(f"\n{'='*72}\nTOP TOKENS - paling sering muncul di top-{TOP_K} attribution\n{'='*72}")
print(f"\nFor 'hate' predictions (n={int((pred_arr==1).sum())}):")
for tok, ct in hate_top_tokens.most_common(30):
    print(f"  {tok:30s} {ct:4d}")
print(f"\nFor 'not_hate' predictions (n={int((pred_arr==0).sum())}):")
for tok, ct in nothate_top_tokens.most_common(30):
    print(f"  {tok:30s} {ct:4d}")


TOP TOKENS - paling sering muncul di top-5 attribution

For 'hate' predictions (n=357):
  political                        87
  individual                       66
  [                                52
  institut                         50
  group                            45
  orig                             44
  ##ion                            41
  seorang                          33
  gender                           30
  national                         27
  ,                                23
  publik                           19
  kritik                           19
  sub                              16
  satir                            14
  perilaku                         11
  ]                                11
  negara                           11
  ketika                           10
  media                            10
  humor                             9
  kontras                           8
  ironi                             8
  pria                              8

## CELL 19 — Save Top-K Analysis

In [19]:
top_payload = {
    "config": {"top_k": TOP_K},
    "per_sample_top_tokens": [
        {
            "idx":        int(all_results[i]["idx"]),
            "pred_class": int(all_results[i]["pred_class"]),
            "true_label": int(all_results[i]["true_label"]),
            "top_tokens": [{"token": t, "l2_score": float(s)} for t, s in top_tokens_per_sample[i]],
        }
        for i in range(len(all_results))
    ],
    "aggregate_hate_top30":     hate_top_tokens.most_common(30),
    "aggregate_nothate_top30":  nothate_top_tokens.most_common(30),
}
with open(OUTPUT_DIR / "ig_top_tokens.json", "w") as f:
    json.dump(top_payload, f, ensure_ascii=False, indent=2)
print(f"Saved: ig_top_tokens.json")

Saved: ig_top_tokens.json


## CELL 20 — Summary

In [20]:
final_summary = {
    "exp_id":     CONFIG["exp_id"],
    "ckpt_path":  str(CKPT_PATH),
    "n_samples":  len(all_results),
    "ig_config": {
        "n_steps":         CONFIG["ig_n_steps"],
        "internal_bs":     CONFIG["ig_internal_bs"],
        "target_mode":     CONFIG["ig_target_mode"],
        "baseline":        "pad_token",
    },
    "convergence_delta": {
        "mean": float(np.mean(np.abs(delta_arr))),
        "p95":  float(np.percentile(np.abs(delta_arr), 95)),
    },
    "model_test_accuracy": float((pred_arr == label_arr).mean()),
    "elapsed_min":         elapsed / 60,
    "outputs": {
        "ig_attributions.npz": "shape (n, 128) untuk token_l2 & token_sum",
        "ig_tokens.json":      "decoded tokens per sample",
        "ig_top_tokens.json":  "top-K tokens analysis & aggregate",
    },
}

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)
with open(OUTPUT_DIR / "final_summary.json", "w") as f:
    json.dump(final_summary, f, indent=2)

print(f"\n{'='*72}\nKOMPONEN 1 (IG) DONE - KIRIMKAN BLOK INI KE CHAT\n{'='*72}")
print(f"Experiment        : {CONFIG['exp_id']}")
print(f"Samples processed : {len(all_results)}")
print(f"Test accuracy     : {(pred_arr == label_arr).mean():.4f}")
print(f"Conv delta (mean) : {np.mean(np.abs(delta_arr)):.4f}")
print(f"Elapsed           : {elapsed/60:.1f} min")
print(f"\nArtifacts saved to: {OUTPUT_DIR}")
print(f"  - ig_attributions.npz  ({(token_l2_arr.nbytes + token_sum_arr.nbytes)/1e6:.1f} MB)")
print(f"  - ig_tokens.json")
print(f"  - ig_top_tokens.json")
print(f"  - final_summary.json")


KOMPONEN 1 (IG) DONE - KIRIMKAN BLOK INI KE CHAT
Experiment        : 2A.1_integrated_gradients
Samples processed : 1005
Test accuracy     : 0.8149
Conv delta (mean) : 0.5205
Elapsed           : 27.6 min

Artifacts saved to: /kaggle/working/xai_2A1_integrated_gradients
  - ig_attributions.npz  (1.0 MB)
  - ig_tokens.json
  - ig_top_tokens.json
  - final_summary.json
